# 02 — Compare Methods

You now have two distance functions. One is fast and wrong. One is correct.

This notebook puts them side by side — not just to show they differ, but to show *how much* they differ, *where* they differ most, and what drives the gap. By the end you should have a reliable instinct for when each approach is appropriate.

In [ ]:
import math

def euclidean_distance(p1, p2):
    """Straight-line distance in degrees. Not a real distance."""
    return math.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)

def haversine_km(p1, p2):
    """Great-circle distance in kilometers."""
    R = 6371.0
    lat1, lon1 = math.radians(p1[1]), math.radians(p1[0])
    lat2, lon2 = math.radians(p2[1]), math.radians(p2[0])
    d_lat = lat2 - lat1
    d_lon = lon2 - lon1
    a = math.sin(d_lat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(d_lon / 2)**2
    return 2 * R * math.asin(math.sqrt(a))

## 1. Side-by-Side Comparison

The table below runs both methods across a range of pairs — from a few kilometers apart to intercontinental. The Euclidean column is in degrees; the Haversine column is in kilometers. They cannot be directly compared in value, but the pattern of divergence is clear.

In [ ]:
pairs = [
    ("On-base (< 1 km)",        (-98.490, 33.910), (-98.495, 33.915)),
    ("Same city (~10 km)",       (-98.49, 33.91),   (-98.40, 33.98)),
    ("City to city (~130 km)",   (-98.5,  33.8),    (-97.2,  34.1)),
    ("WF → Dallas (~200 km)",    (-98.49, 33.91),   (-96.80, 32.78)),
    ("Dallas → Houston (~360 km)",(-96.80, 32.78),  (-95.37, 29.76)),
    ("Dallas → Denver (~1300 km)",(-96.80, 32.78),  (-104.99, 39.74)),
    ("Dallas → New York (~2200 km)",(-96.80, 32.78),(-74.01, 40.71)),
    ("Dallas → London (~8000 km)",(-96.80, 32.78),  (-0.13, 51.51)),
]

KM_PER_DEG = 111.32   # rough conversion at ~33°N for display only

print(f"{'Pair':<30}  {'Euclidean (°)':>14}  {'~km equiv':>10}  {'Haversine (km)':>15}")
print("-" * 76)
for label, a, b in pairs:
    e = euclidean_distance(a, b)
    h = haversine_km(a, b)
    approx = e * KM_PER_DEG
    print(f"{label:<30}  {e:>13.4f}°  {approx:>9.1f}   {h:>14.1f} km")

## 2. Error Growth

To compare the methods directly, we need a common unit. The Euclidean value in degrees can be scaled by `~111.32 km/°` (the rough latitude-to-km conversion) to get an approximate kilometer estimate — then compared against Haversine as the ground truth.

The percentage error shows how badly the scaling assumption breaks down at larger distances.

In [ ]:
def pct_error(euclidean_deg, haversine_km_val):
    """
    Convert Euclidean degrees to a km estimate, compare to Haversine truth.
    Note: this conversion is approximate and latitude-dependent.
    """
    euclidean_approx_km = euclidean_deg * KM_PER_DEG
    return abs(euclidean_approx_km - haversine_km_val) / haversine_km_val * 100


print(f"{'Pair':<30}  {'Haversine':>12}  {'Eucl. approx':>13}  {'Error':>8}")
print("-" * 70)
for label, a, b in pairs:
    e = euclidean_distance(a, b)
    h = haversine_km(a, b)
    err = pct_error(e, h)
    flag = "  ← SIGNIFICANT" if err > 5 else ""
    print(f"{label:<30}  {h:>10.1f} km  {e * KM_PER_DEG:>11.1f} km  {err:>7.1f}%{flag}")

The error is small at short range and grows as the distance increases. Two sources combine to produce it:

1. **Longitude compression** — the Euclidean formula uses a fixed `111.32 km/°` for both axes, but longitude degrees are shorter than latitude degrees at any latitude above 0°
2. **Curvature accumulation** — over long distances, the Earth's surface curves away from the flat-plane assumption, adding error that compounds with distance

These are not the same thing, and both are always present to some degree.

## 3. The Latitude Effect

Now isolate the longitude compression factor. Take one fixed degree of longitude separation — `Δlon = 1.0°` — and hold latitude constant per row. The Euclidean distance stays `1.0°` every time. The Haversine distance changes dramatically.

In [ ]:
latitudes = [0, 10, 20, 30, 40, 50, 60, 70, 80]

print(f"{'Latitude':>10}  {'Euclidean (°)':>14}  {'Haversine (km)':>15}  {'Error':>8}")
print("-" * 56)
for lat in latitudes:
    a = (-10.0, float(lat))
    b = ( -9.0, float(lat))   # exactly 1° east, same latitude
    e = euclidean_distance(a, b)
    h = haversine_km(a, b)
    err = pct_error(e, h)
    print(f"{lat:>9}°  {e:>13.4f}°  {h:>14.2f} km  {err:>7.1f}%")

The Euclidean value is identical in every row — `1.0°`. The Haversine value drops from ~111 km at the equator to ~19 km near 80°N. The error is zero at the equator (where the axes happen to be the same scale) and climbs steadily toward the poles.

This is not a rounding error or a precision issue. It is a geometric fact about what latitude and longitude mean on a sphere.

## 4. Visualize the Difference

Draw three lines on a single map: one short local pair, one medium pair, and one long pair. Each line is labeled with both distances — degree-based and kilometer-based — so the divergence is visible in the context of the actual geography.

In [ ]:
from ipyleaflet import Map, GeoJSON

showcase = [
    ("Short",  (-98.49, 33.91), (-97.2,  34.1),  "#2a9d8f"),
    ("Medium", (-98.49, 33.91), (-96.80, 32.78),  "#e9c46a"),
    ("Long",   (-98.49, 33.91), (-74.01, 40.71),  "#e63946"),
]

features = []
for label, a, b, color in showcase:
    e = euclidean_distance(a, b)
    h = haversine_km(a, b)
    features.append({
        "type": "Feature",
        "geometry": {"type": "LineString", "coordinates": [list(a), list(b)]},
        "properties": {
            "label": label,
            "euclidean_deg": round(e, 4),
            "haversine_km":  round(h, 1),
        }
    })
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": list(b)},
        "properties": {"name": label}
    })
    print(f"{label:<8}  Euclidean: {e:.4f}°   Haversine: {h:.1f} km")

# Add base point
features.append({
    "type": "Feature",
    "geometry": {"type": "Point", "coordinates": [-98.49, 33.91]},
    "properties": {"name": "Base"}
})

fc = {"type": "FeatureCollection", "features": features}

m = Map(center=(36.0, -88.0), zoom=4)
m.add(GeoJSON(
    data=fc,
    style={"color": "#457b9d", "weight": 2},
    hover_style={"color": "#e63946"}
))
m

---

## Exercise A — Compute Percentage Error

For each pair below, compute the Euclidean distance in degrees, convert it to an approximate km estimate using `111.32 km/°`, and compute the percentage error against the Haversine result. Print a table sorted by error, largest to smallest.

In [ ]:
exercise_pairs = [
    ("Tinker AFB → NAS Fort Worth",  (-97.37, 35.39), (-97.04, 32.85)),
    ("Dallas → Chicago",             (-96.80, 32.78), (-87.63, 41.88)),
    ("Dallas → Mexico City",         (-96.80, 32.78), (-99.13, 19.43)),
    ("OKC → Tulsa",                  (-97.52, 35.47), (-95.99, 36.15)),
    ("Dallas → Buenos Aires",        (-96.80, 32.78), (-58.38, -34.61)),
    ("Wichita Falls → Abilene",      (-98.49, 33.91), (-99.73, 32.45)),
]

# your code here
# expected output: table sorted by error % descending

## Exercise B — Find the Crossover Point

Starting from the base point `(-98.49, 33.91)`, step east in 0.5° increments and compute both distances for each step. Find the approximate separation (in km) where the Euclidean error first exceeds **5%**.

Print the crossover row.

In [ ]:
base = (-98.49, 33.91)

# your code here
# step east from base in 0.5° increments up to 20°
# for each step: compute euclidean_distance, haversine_km, pct_error
# stop and print the first row where error > 5%

## Exercise C — Explain the Errors

Look at your Exercise A results. Answer the following in plain English (comments or print statements are fine):

1. Which pair had the highest error? Why — is it mostly about distance, latitude change, or both?
2. Which pair had the lowest error? What makes it a "safe" case for Euclidean approximation?
3. The Dallas → Mexico City and Dallas → Chicago pairs are at similar distances. Do they have similar errors? If not, why not?

In [ ]:
# your answers here

---

## Check Your Understanding

A teammate writes a function to find all points within a given radius and uses Euclidean distance to avoid importing anything:

```python
def within_radius(base, points, radius_km):
    threshold = radius_km / 111.32   # convert km to degrees
    return [p for p in points if euclidean_distance(base, p) <= threshold]
```

They test it in Texas on a small local dataset and it seems to work. Then the dataset expands to include points in Canada and the results are wrong — some points inside the radius are excluded, others outside are included.

**Question:** Identify two distinct reasons why this function produces incorrect results for a dataset spanning Texas to Canada. For each reason, state whether it would cause false positives (non-members returned), false negatives (members excluded), or both.

```python
# your answer here
```


---

## Next

In [03 — Distance Applications](./03-Distance_Applications.ipynb), we put `haversine_km` to work on real spatial problems: nearest-neighbor search, radius filtering, and distance-from-click interactions.